## Bring in detailed 1km population from WorldPOP
This script worldpop data from 2015-2030 at the 1km granularity level. 
It uses the census.gov tiger file to cut out the state of california
then it transforms the .tif file from pixels to coords
after that it increases granularity to match the URMA 2.5km granularity using the center of each cell to sum up the population for every worldpop cell where that center is within an urma cell.

This is imperfect, but close enough for understanding ratios for weather

ChatGPT was used to:
- help decide on method
- code and debug
- undersatnd risks and beneifts of each method
- support coding in QC checking

Methods: talk to chat GPT about challenge, get suggestions. google and undersatnd suggestions, test small data points first, qc check, run larger query, check and qc. QC and understand methods for transfering from .tif to grid file

In [1]:
!pip install rioxarray

  Using cached rioxarray-0.21.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached rasterio-1.5.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (8.6 kB)
  Using cached xarray-2025.11.0-py3-none-any.whl.metadata (12 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
Using cached rioxarray-0.21.0-py3-none-any.whl (64 kB)
Using cached xarray-2025.11.0-py3-none-any.whl (1.4 MB)
Using cached rasterio-1.5.0-cp314-cp314-macosx_14_0_arm64.whl (22.8 MB)
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
Using cached affine-2.4.0-py3-none-any.whl (15 kB)
  Attempting uninstall: xarray90m╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [rasterio]
    Found existing installation: xarray 2026.2.0━━━━━━━━━━━━━━ 2/5 [rasterio]
    Uninstalling xarray-2026.2.0:m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [rasterio]
      Successfully uninstalled xarray-2026.2.0━━━━━━━━━━━━━━━━ 2/5 [rasterio]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [rioxarray]/5 [xarray]
ER

In [4]:
from pathlib import Path
import requests
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import zarr as xr


In [5]:
import zarr
print(f"Current zarr version: {zarr.__version__}")

Current zarr version: 3.1.5


In [6]:
from pathlib import Path
import requests, zipfile, io

url = "https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip"
out_dir = Path("data/boundaries/tiger_state_2023")
out_dir.mkdir(parents=True, exist_ok=True)

r = requests.get(url, timeout=120)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall(out_dir)

print("✅ Unzipped to:", out_dir)
print("Files:", [p.name for p in out_dir.iterdir()])

✅ Unzipped to: data/boundaries/tiger_state_2023
Files: ['tl_2023_us_state.shp.iso.xml', 'tl_2023_us_state.dbf', 'tl_2023_us_state.cpg', 'tl_2023_us_state.shp', 'tl_2023_us_state.shx', 'tl_2023_us_state.prj', 'tl_2023_us_state.shp.ea.iso.xml']


In [5]:
import geopandas as gpd
from pathlib import Path

shp = Path("data/boundaries/tiger_state_2023/tl_2023_us_state.shp")
gdf = gpd.read_file(shp)

ca = gdf[gdf["STATEFP"] == "06"].to_crs("EPSG:4326")

out_geojson = Path("data/boundaries/ca.geojson")
out_geojson.parent.mkdir(parents=True, exist_ok=True)
ca.to_file(out_geojson, driver="GeoJSON")

print("✅ Wrote:", out_geojson)


✅ Wrote: data/boundaries/ca.geojson


In [6]:
from pathlib import Path
import requests

url = "https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2015/USA/v1/1km_ua/constrained/usa_pop_2015_CN_1km_R2025A_UA_v1.tif"

raw_tif = Path("tmp_worldpop/usa_pop_2015_1km_ua.tif")
raw_tif.parent.mkdir(parents=True, exist_ok=True)

with requests.get(url, stream=True, timeout=180) as r:
    r.raise_for_status()
    with open(raw_tif, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("✅ Downloaded:", raw_tif, "Size (GB):", raw_tif.stat().st_size / 1e9)


✅ Downloaded: tmp_worldpop/usa_pop_2015_1km_ua.tif Size (GB): 0.036982915


In [7]:
## Cookie cutter out cali cali

from pathlib import Path
import geopandas as gpd
import rasterio ## opens .tif files
from rasterio.mask import mask

ca_geom_path = Path("data/boundaries/ca.geojson")
ca_tif = Path("tmp_worldpop/ca_pop_2015_1km_ua.tif")

gdf = gpd.read_file(ca_geom_path)

with rasterio.open(raw_tif) as src: ### open worldpop
    gdf = gdf.to_crs(src.crs)  ## shift tiger file counties to match shape of worldpop
    geoms = [geom.__geo_interface__ for geom in gdf.geometry] ## changing TIGER CA polygon to geoJSON format so we can do stuff

    out_img, out_transform = mask(src, geoms, crop=True, nodata=src.nodata)
    ## take the src(worldPOP) and TiGER counties, crop, nodata outside of polygon

    out_meta = src.meta.copy() ##make a copy of the new data
    out_meta.update({ ##add some fun info
        "height": out_img.shape[1],
        "width": out_img.shape[2],
        "transform": out_transform
    })

ca_tif.parent.mkdir(parents=True, exist_ok=True) #make a path to put this stuff
with rasterio.open(ca_tif, "w", **out_meta) as dst:
    dst.write(out_img) ##write that

    ## save our fresh california cookie here:

print("✅ Clipped CA raster:", ca_tif, "Size (MB):", ca_tif.stat().st_size / 1e6)


✅ Clipped CA raster: tmp_worldpop/ca_pop_2015_1km_ua.tif Size (MB): 5.670327


In [8]:
ca_tif

PosixPath('tmp_worldpop/ca_pop_2015_1km_ua.tif')

In [9]:
raw_tif.unlink()
print("🧹 Deleted raw USA file:", raw_tif)

🧹 Deleted raw USA file: tmp_worldpop/usa_pop_2015_1km_ua.tif


In [10]:
### confirming ca_tif file has information we need.

import rasterio
import numpy as np

with rasterio.open(ca_tif) as src:
    band = src.read(1, masked=True)
    print("CRS:", src.crs)
    print("Shape:", band.shape)
    print("Bounds:", src.bounds)
    print("Min/Max:", float(band.min()), float(band.max()))
    print("Nonzero pixels:", int(np.count_nonzero(band.filled(0))))

CRS: EPSG:4326
Shape: (1139, 1243)
Bounds: BoundingBox(left=-124.48333355539998, bottom=32.5250002059, right=-114.12500026349998, top=42.0166668346)
Min/Max: 0.0 28014.228515625
Nonzero pixels: 222995


In [11]:
## confirm population within cali cookie cutter raster

import rasterio
import numpy as np
from pathlib import Path

ca_tif = Path("tmp_worldpop/ca_pop_2015_1km_ua.tif")  # adjust if needed

with rasterio.open(ca_tif) as src:
    band = src.read(1, masked=True).filled(0)

total_pop = float(band.sum())
print(f"Total population (raster sum): {total_pop:,.0f}")


Total population (raster sum): 40,436,988


## PULL IN URMA

In [14]:
## Pull in URMA
import xarray as xr
directory = '01_URMA_RAW/URMA_RAW_2018_test.zarr'
ds = xr.open_zarr(directory)
ds

<xarray.Dataset> Size: 113MB
Dimensions:                (date: 14, y: 535, x: 457)
Coordinates:
  * date                   (date) datetime64[ns] 112B 2018-01-01 ... 2018-01-15
    atmosphereSingleLayer  float64 8B ...
    latitude               (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
    longitude              (y, x) float64 2MB dask.array<chunksize=(535, 457), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    cloud_cover_pct        (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_afternoon_k        (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    dpt_morning_k          (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    gribfile_projection    float64 8B ...
    spfh_peak_kgkg         (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_max_k              (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    t2m_min_k              (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_low_ms            (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>
    wind_peak_ms           (date, y, x) float32 14MB dask.array<chunksize=(1, 535, 457), meta=np.ndarray>

In [15]:
## IMPORTANT: FIX COORDS

ds = ds.assign_coords(
    longitude = ((ds.longitude + 180) % 360) - 180
)


In [16]:
# If latitude/longitude are coords or data variables on your URMA dataset `ds`
lat_min = float(ds["latitude"].min().compute())
lat_max = float(ds["latitude"].max().compute())
lon_min = float(ds["longitude"].min().compute())
lon_max = float(ds["longitude"].max().compute())

print("URMA lat:", lat_min, "to", lat_max)
print("URMA lon:", lon_min, "to", lon_max)


URMA lat: 30.399658978941538 to 43.89945484380344
URMA lon: -127.48243310481524 to -112.20237479820963


In [17]:
%%time
from pathlib import Path
import numpy as np
import rasterio
from scipy.spatial import cKDTree

ca_tif = Path("tmp_worldpop/ca_pop_2015_1km_ua.tif")  

# --- 1) Build KDTree on URMA cell centers ---
lat_u = ds["latitude"].values
lon_u = ds["longitude"].values

# Flatten (Ncells, 2): (lon, lat) ->> created searchable spatial index
urma_pts = np.column_stack([lon_u.ravel(), lat_u.ravel()])
tree = cKDTree(urma_pts)

# Accumulator on flattened URMA grid
acc = np.zeros(lat_u.size, dtype=np.float64) ##-> make a 1d tree

# --- 2) Read WorldPop, get pixel center lon/lat for nonzero pixels ---
with rasterio.open(ca_tif) as src:
    pop = src.read(1, masked=True).filled(0).astype(np.float64)
    t = src.transform  # affine transform from pixel index to latlon -> get info

rows, cols = np.nonzero(pop > 0)
vals = pop[rows, cols]

# Convert row/col → lon/lat of pixel centers (vectorized)
# For north-up rasters in EPSG:4326: lon = c + (col+0.5)*a ; lat = f + (row+0.5)*e
lons = t.c + (cols + 0.5) * t.a ##c -> x coord, 0.5 moves to center
lats = t.f + (rows + 0.5) * t.e ## f -> t coord, 0.5 moves to center from upper left

# --- 3) Nearest URMA cell for each pop pixel, accumulate SUM ---
# Chunk to keep memory chill
chunk = 200_000
for i in range(0, len(vals), chunk):
    j = min(i + chunk, len(vals))
    _, idx = tree.query(np.column_stack([lons[i:j], lats[i:j]]), k=1)
    np.add.at(acc, idx, vals[i:j])

pop_on_urma = acc.reshape(lat_u.shape).astype("float32")

# --- 4) Attach to ds (for 2015 baseline) ---
ds = ds.assign_coords(pop_year=2015)
ds["population"] = (("y", "x"), pop_on_urma)
ds["population"].attrs.update({
    "source": "WorldPop R2025A 1km_ua constrained (CA clipped)",
    "year": 2015,
    "aggregation": "nearest_urma_cell_center_sum",
    "units": "persons"
})

# --- 5) Sanity check: sums should match closely ---
print("WorldPop CA total (from tif):", float(vals.sum()))
print("URMA-grid total:", float(ds['population'].sum().values))


WorldPop CA total (from tif): 40436986.057291195
URMA-grid total: 40436988.0
CPU times: user 248 ms, sys: 13.3 ms, total: 262 ms
Wall time: 262 ms


In [18]:
import shutil
shutil.rmtree("processed/worldpop_population_on_urma.zarr", ignore_errors=True)

In [19]:
## PIPELINE

In [20]:
### Nearest Neighbor spatial aggregation
## Assuptions URMA grid is 2.5km, POP grid is 1km
## How we aligned: each 1km pixel belongs to the URMA cell who's center is the closest
## Sum the population to get the closest
### WHY? URMA is coarser so we can do this + we are using this for estimations, not estimating sub grids

In [23]:
import time
from pathlib import Path
import requests
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from scipy.spatial import cKDTree
import xarray as xr

# -----------------------
# Config
# -----------------------
START_YEAR ='2019' 
END_YEAR = '2021'
CA_GEOJSON = Path("data/boundaries/ca.geojson")
WORK_DIR = Path("tmp_worldpop")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ZARR_OUT = Path(f"processed/worldpop_population_{START_YEAR}_{END_YEAR}.zarr")
ZARR_OUT.parent.mkdir(parents=True, exist_ok=True)

URL_TMPL = (
    "https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/{year}/USA/v1/"
    "1km_ua/constrained/usa_pop_{year}_CN_1km_R2025A_UA_v1.tif"
)

YEARS = list(range(int(START_YEAR), int(END_YEAR)))
                                                

# -----------------------
# Build KDTree once (URMA grid centers)
# -----------------------
lat_u = ds["latitude"].values
lon_u = ds["longitude"].values
urma_pts = np.column_stack([lon_u.ravel(), lat_u.ravel()])
tree = cKDTree(urma_pts)
n_cells = lat_u.size

# Load CA polygon once
ca_gdf = gpd.read_file(CA_GEOJSON)

def download_tif(year: int, out_path: Path) -> Path:
    url = URL_TMPL.format(year=year)
    t0 = time.time()
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    dt = time.time() - t0
    print(f"⬇️ {year} download: {dt:.1f}s  size={out_path.stat().st_size/1e6:.0f}MB")
    return out_path

def clip_to_ca(raw_tif: Path, ca_tif: Path) -> Path:
    with rasterio.open(raw_tif) as src:
        gdf = ca_gdf.to_crs(src.crs)
        geoms = [geom.__geo_interface__ for geom in gdf.geometry]

        out_img, out_transform = mask(src, geoms, crop=True, nodata=src.nodata)

        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform
        })

    with rasterio.open(ca_tif, "w", **out_meta) as dst:
        dst.write(out_img)

    return ca_tif

def map_ca_raster_to_urma(ca_tif: Path, chunk: int = 200_000) -> np.ndarray:
    acc = np.zeros(n_cells, dtype=np.float64)

    with rasterio.open(ca_tif) as src:
        pop = src.read(1, masked=True).filled(0).astype(np.float64)
        t = src.transform

    rows, cols = np.nonzero(pop > 0)
    vals = pop[rows, cols]

    lons = t.c + (cols + 0.5) * t.a
    lats = t.f + (rows + 0.5) * t.e

    ### acutally using nearest neighbor and the KDTree to find matching cell centers between URMA and worldpop counties
    for i in range(0, len(vals), chunk):
        j = min(i + chunk, len(vals))
        _, idx = tree.query(np.column_stack([lons[i:j], lats[i:j]]), k=1)
        np.add.at(acc, idx, vals[i:j])

    return acc.reshape(lat_u.shape).astype("float32") ## pull out the pop

def append_year_to_zarr(pop_yx: np.ndarray, year: int, zarr_path):
    da = xr.DataArray(
        pop_yx,
        dims=("y", "x"),
        name="population",
        attrs={
            "source": "WorldPop R2025A 1km_ua constrained (CA clipped)",
            "aggregation": "nearest_urma_cell_center_sum",
            "units": "persons"
        },
    )
    ds_out = da.to_dataset().expand_dims(year=[np.int32(year)])

    if zarr_path.exists():
        # ✅ append WITHOUT encoding
        ds_out.to_zarr(zarr_path, mode="a", append_dim="year", consolidated=False)
    else:
        # ✅ first write WITH encoding (define chunks once)
        enc = {"population": {"chunks": (1, 256, 256), "dtype": "float32"}}
        ds_out.to_zarr(zarr_path, mode="w", consolidated=False, encoding=enc)
# -----------------------
# Run
# -----------------------
for year in YEARS:
    print(f"\n==== {year} ====")
    raw = WORK_DIR / f"usa_pop_{year}_1km_ua.tif"
    ca  = WORK_DIR / f"ca_pop_{year}_1km_ua.tif"

    try:
        download_tif(year, raw)
        t0 = time.time()
        clip_to_ca(raw, ca)
        print(f"✂️ {year} clip: {time.time()-t0:.1f}s  size={ca.stat().st_size/1e6:.0f}MB")

        t0 = time.time()
        pop_yx = map_ca_raster_to_urma(ca)
        print(f"🧭 {year} map→URMA: {time.time()-t0:.2f}s  total={float(pop_yx.sum()):,.0f}")

        append_year_to_zarr(pop_yx, year, ZARR_OUT)
        print(f"💾 appended to: {ZARR_OUT}")

    finally:
        # Delete intermediates
        for p in [raw, ca]:
            try:
                if p.exists():
                    p.unlink()
            except Exception:
                pass

print("\n✅ Done:", ZARR_OUT)


==== 2019 ====
⬇️ 2019 download: 108.2s  size=37MB
✂️ 2019 clip: 0.1s  size=6MB
🧭 2019 map→URMA: 0.19s  total=42,237,872
💾 appended to: processed/worldpop_population_2019_2021.zarr

==== 2020 ====
⬇️ 2020 download: 118.7s  size=37MB
✂️ 2020 clip: 0.1s  size=6MB
🧭 2020 map→URMA: 0.18s  total=42,655,288
💾 appended to: processed/worldpop_population_2019_2021.zarr

✅ Done: processed/worldpop_population_2019_2021.zarr


In [26]:
import xarray as xr

pop = xr.open_zarr(f"processed/worldpop_population_2019_2021.zarr", consolidated=False)

totals = pop["population"].sum(dim=("y","x")).compute()
totals


<xarray.DataArray 'population' (year: 2)> Size: 8B
array([4.2237876e+07, 4.2655292e+07], dtype=float32)
Coordinates:
  * year     (year) int32 8B 2019 2020
Attributes:
    aggregation:  nearest_urma_cell_center_sum
    source:       WorldPop R2025A 1km_ua constrained (CA clipped)
    units:        persons

In [27]:
print(pop.coords)
print(pop.dims)


Coordinates:
  * year     (year) int32 8B 2019 2020
FrozenMappingWarningOnValuesAccess({'year': 2, 'y': 535, 'x': 457})


In [28]:
pop

<xarray.Dataset> Size: 2MB
Dimensions:     (year: 2, y: 535, x: 457)
Coordinates:
  * year        (year) int32 8B 2019 2020
Dimensions without coordinates: y, x
Data variables:
    population  (year, y, x) float32 2MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>

In [ ]:
mn = pop["population"].min().compute().item()
mx = pop["population"].max().compute().item()
has_nan = pop["population"].isnull().any().compute().item()

print("Min:", mn)
print("Max:", mx)
print("Any NaNs?", has_nan)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vals = pop["population"].values.flatten()

plt.hist(np.log10(vals + 1), bins=100)
plt.title("Log10 Population per Grid Cell")
plt.xlabel("log10(population + 1)")
plt.ylabel("Count")
plt.show()


In [ ]:
growth = totals.diff("year")
for y, v in zip(growth["year"].values, growth.values):
    print(int(y), f"{float(v):+,.0f}")


In [ ]:
import matplotlib.pyplot as plt

pop["population"].sel(year=2015).plot(figsize=(8,6))
plt.title("Population on URMA Grid (201)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

pop["population"].sel(year=2015).plot(figsize=(8,6))
plt.title("Population on URMA Grid (2015)")
plt.show()


In [ ]:
pop

In [ ]:
pop

In [ ]:
# count cells with population > 0 but no county assigned
mask = (pop["population"].sel(year=2019) > 0) & (ds["county"].isnull())
mask.sum().compute()
